# XHS Sample Pool Data Processing

用于小红书样本池的数据清洗、检查、标注辅助和后续特征处理。

In [ ]:
from pathlib import Path
import json
import re
import math
import statistics
from datetime import datetime
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)


In [ ]:
WORKSPACE_ROOT = Path(r'D:/codex/project-001')


def find_sample_pool() -> Path:
    """查找样本池 CSV，优先使用当前项目里的 data-pool 文件夹。"""
    search_roots = []

    if WORKSPACE_ROOT.exists():
        search_roots.append(WORKSPACE_ROOT)

    cwd = Path.cwd().resolve()
    search_roots.extend([cwd, *cwd.parents])

    seen_roots = []
    for root in search_roots:
        if root not in seen_roots and root.exists():
            seen_roots.append(root)

    preferred_parts = ('Database', 'Datapool', 'data-pool', 'xhs_sample_pool_v1.csv')
    for root in seen_roots:
        candidate = root.joinpath(*preferred_parts)
        if candidate.exists():
            return candidate

    for root in seen_roots:
        matches = sorted(root.rglob('xhs_sample_pool_v1.csv'))
        if matches:
            return matches[0]

    raise FileNotFoundError('没有找到 xhs_sample_pool_v1.csv，请确认样本池文件在项目目录内。')


SAMPLE_POOL_PATH = find_sample_pool()
DATA_POOL_DIR = SAMPLE_POOL_PATH.parent
PROJECT_ROOT = WORKSPACE_ROOT if WORKSPACE_ROOT.exists() else DATA_POOL_DIR.parent

print(f'样本池路径：{SAMPLE_POOL_PATH}')
PROJECT_ROOT, DATA_POOL_DIR, SAMPLE_POOL_PATH

In [ ]:
df = pd.read_csv(SAMPLE_POOL_PATH)
df.head()

In [ ]:
# 删除 images 为空或 keep_status 标为“剔除”的样本，删除不需要的列，并保存回样本池 CSV。
before_count = len(df)
drop_columns = ['videos', 'author_avatar', 'cover_type','shares']

df['images'] = df['images'].fillna('').astype(str).str.strip()
image_valid_mask = df['images'] != ''
keep_status = df.get('keep_status', pd.Series('', index=df.index)).fillna('').astype(str).str.strip()
excluded_mask = keep_status == '剔除'
removed_missing_images = int((~image_valid_mask).sum())
removed_excluded_samples = int((image_valid_mask & excluded_mask).sum())
df = df[image_valid_mask & ~excluded_mask].copy()
df = df.drop(columns=drop_columns, errors='ignore')

after_count = len(df)
removed_count = before_count - after_count
dropped_columns = [column for column in drop_columns if column not in df.columns]

df.to_csv(SAMPLE_POOL_PATH, index=False, encoding='utf-8-sig')

print(f'清洗前：{before_count} 条')
print(f'删除 images 为空：{removed_missing_images} 条')
print(f'删除 keep_status 为剔除：{removed_excluded_samples} 条')
print(f'清洗后：{after_count} 条')
print(f'已删除列：{dropped_columns}')

In [ ]:
# Clean title and content columns.
# - content: extract inline tags to tags column, remove inline tags, remove non-content notes, replace line breaks with two spaces.
# - title: normalize surrounding whitespace only.

explain_patterns = [
    re.compile(r'\u539f\u521b\s*ip?\s*\u7981?\s*\u6284\u88ad', re.I),
    re.compile(r'\u7981\s*\u6284\u88ad', re.I),
    re.compile(r'\u7981\s*\u4e8c\u6539', re.I),
    re.compile(r'\u539f\u521b\s*ip?\s*$', re.I),
    re.compile(r'\u4e8c\u6539\s*\u6284\u88ad|\u6284\u88ad\s*\u4e8c\u6539'),
    re.compile(r'\u4e3b\u8981\u662f\u5bf9\u4e0a\u6b21.*?(\u6392\u7248|\u7248\u672c).*?(\u6539\u8fdb|\u91cd\u65b0\u62cd)'),
    re.compile(r'\u91cd\u65b0\u62cd\u4e86?\u4e00\u4e2a\u7248\u672c'),
    re.compile(r'\u6700\u8fd1\u6709\u6863\u671f|\u53ef\u7ea6\u7a3f|\u7ea6\u7a3f'),
]

trim_chars = ' -_.,:;!?|/\\()<>"\'&+~%' + '\uFF0C\u3002\uFF1A\uFF1B\uFF01\uFF1F\uFF5C\uFF08\uFF09\u3010\u3011\u300A\u300B\u201C\u201D\u00B7\u3001\uFF5E'
tag_punctuation = '#\\s,.;:!/?~|\\\\\uFF0C\u3002\uFF1B\uFF1A\u3001\uFF01\uFF1F\uFF5E'

def normalize_text_spacing(value):
    if pd.isna(value):
        return ''
    text = str(value).replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'[\t ]+', ' ', text)
    return text.strip()

def is_all_tags(text: str) -> bool:
    compact = re.sub(r'\s+', '', text or '')
    if not compact or '#' not in compact:
        return False
    compact = compact.strip(trim_chars)
    return bool(re.fullmatch(r'(#[^' + tag_punctuation + r']+)+', compact))


def extract_inline_tags(text: str) -> list[str]:
    if not text:
        return []
    tags = []
    for match in re.finditer(r'#[^#\s,.;:!/?~|\\，。；：、！？～]+', str(text)):
        tag = match.group(0).strip()
        if tag and tag not in tags:
            tags.append(tag)
    return tags


def remove_inline_tags(text: str) -> str:
    if not text:
        return ''
    text = re.sub(r'#[^#\s,.;:!/?~|\\，。；：、！？～]+', '', str(text))
    text = re.sub(r'\s{3,}', '  ', text)
    return text.strip()


def merge_tag_values(existing_value, new_tags: list[str]) -> str:
    merged = []
    for item in str(existing_value or '').split(';'):
        item = item.strip()
        if item and item not in merged:
            merged.append(item)
    for tag in new_tags:
        if tag and tag not in merged:
            merged.append(tag)
    return '; '.join(merged)

def clean_content(value):
    if pd.isna(value):
        return ''
    text = str(value).replace('\r\n', '\n').replace('\r', '\n')
    if is_all_tags(text):
        return ''

    kept_lines = []
    for raw_line in text.split('\n'):
        line = raw_line.strip()
        if not line:
            continue
        if is_all_tags(line):
            continue
        if any(pattern.search(line) for pattern in explain_patterns):
            continue
        kept_lines.append(line)

    text = '  '.join(kept_lines)
    text = remove_inline_tags(text)
    text = text.replace('\u3002', '  ')
    text = re.sub(r' {3,}', '  ', text)
    text = text.strip(trim_chars)

    if is_all_tags(text):
        return ''
    return text

before_empty_content = (df['content'].fillna('').astype(str).str.strip() == '').sum()
content_inline_tags = df['content'].fillna('').astype(str).apply(extract_inline_tags)
if 'tags' not in df.columns:
    df['tags'] = ''
df['tags'] = [merge_tag_values(existing, tags) for existing, tags in zip(df['tags'].fillna(''), content_inline_tags)]
df['content'] = df['content'].apply(clean_content)
after_empty_content = (df['content'].fillna('').astype(str).str.strip() == '').sum()

if 'title' in df.columns:
    df['title'] = df['title'].apply(normalize_text_spacing)

df.to_csv(SAMPLE_POOL_PATH, index=False, encoding='utf-8-sig')

print(f'content empty before: {before_empty_content}')
print(f'content empty after: {after_empty_content}')
df[['id', 'title', 'content']].head()


In [ ]:
# Normalize push_time to YYYY/MM/DD with a fixed collection-time anchor.
PUSH_TIME_ANCHOR_DATE = pd.Timestamp('2026-06-18')
EDITED_PREFIX = '\u7f16\u8f91\u4e8e'
DAY_BEFORE_YESTERDAY = '\u524d\u5929'
YESTERDAY = '\u6628\u5929'
TODAY = '\u4eca\u5929'
JUST_NOW = '\u521a\u521a'
DAYS_AGO_SUFFIX = '\u5929\u524d'
HOURS_AGO_SUFFIX = '\u5c0f\u65f6\u524d'
YEAR_MARK = '\u5e74'
MONTH_MARK = '\u6708'
DAY_MARK = '\u65e5'


def format_push_date(value: pd.Timestamp) -> str:
    return value.strftime('%Y/%m/%d')


def infer_push_year(month: int, day: int) -> int:
    candidate_date = pd.Timestamp(
        year=PUSH_TIME_ANCHOR_DATE.year,
        month=month,
        day=day,
    )
    return (
        PUSH_TIME_ANCHOR_DATE.year - 1
        if candidate_date > PUSH_TIME_ANCHOR_DATE
        else PUSH_TIME_ANCHOR_DATE.year
    )


def normalize_push_time(value: object) -> str | None:
    if pd.isna(value) or not str(value).strip():
        return None

    raw_text = str(value).replace(EDITED_PREFIX, '').strip()

    full_date_match = re.search(r'(20\d{2})[/-](\d{1,2})[/-](\d{1,2})', raw_text)
    if full_date_match:
        year, month, day = map(int, full_date_match.groups())
        return format_push_date(pd.Timestamp(year=year, month=month, day=day))

    chinese_full_date_pattern = (
        rf'(20\d{{2}}){YEAR_MARK}(\d{{1,2}}){MONTH_MARK}(\d{{1,2}}){DAY_MARK}'
    )
    chinese_full_date_match = re.search(chinese_full_date_pattern, raw_text)
    if chinese_full_date_match:
        year, month, day = map(int, chinese_full_date_match.groups())
        return format_push_date(pd.Timestamp(year=year, month=month, day=day))

    days_ago_match = re.search(rf'(\d+){DAYS_AGO_SUFFIX}', raw_text)
    if days_ago_match:
        days_ago = int(days_ago_match.group(1))
        return format_push_date(PUSH_TIME_ANCHOR_DATE - pd.Timedelta(days=days_ago))

    relative_day_offsets = {
        DAY_BEFORE_YESTERDAY: 2,
        YESTERDAY: 1,
        TODAY: 0,
        JUST_NOW: 0,
    }
    for relative_phrase, days_ago in relative_day_offsets.items():
        if relative_phrase in raw_text:
            return format_push_date(PUSH_TIME_ANCHOR_DATE - pd.Timedelta(days=days_ago))

    if re.search(rf'\d+{HOURS_AGO_SUFFIX}', raw_text):
        return format_push_date(PUSH_TIME_ANCHOR_DATE)

    chinese_month_day_pattern = rf'(\d{{1,2}}){MONTH_MARK}(\d{{1,2}}){DAY_MARK}'
    chinese_month_day_match = re.search(chinese_month_day_pattern, raw_text)
    if chinese_month_day_match:
        month, day = map(int, chinese_month_day_match.groups())
        return format_push_date(
            pd.Timestamp(year=infer_push_year(month, day), month=month, day=day)
        )

    dashed_month_day_match = re.search(r'(?<!\d)(\d{1,2})-(\d{1,2})(?!\d)', raw_text)
    if dashed_month_day_match:
        month, day = map(int, dashed_month_day_match.groups())
        return format_push_date(
            pd.Timestamp(year=infer_push_year(month, day), month=month, day=day)
        )

    return None


normalized_push_time = df['push_time'].apply(normalize_push_time)
unresolved_push_time = df.loc[normalized_push_time.isna(), 'push_time']

if not unresolved_push_time.empty:
    raise ValueError(
        'Unresolved push_time values: '
        + ', '.join(unresolved_push_time.astype(str).head(10))
    )

df['push_time'] = normalized_push_time
df.to_csv(SAMPLE_POOL_PATH, index=False, encoding='utf-8-sig')

print(f'push_time normalized: {len(df)} rows')
print(f'anchor date: {format_push_date(PUSH_TIME_ANCHOR_DATE)}')




In [ ]:
df.info()